## 1. Load and Inspect the Data

We begin by loading the dataset and converting the `timestamp` column into a datetime objec.

The data is the sorted chronologically, becuase we must respect the temprotal order of observations to avoid data leakage into our train/test split.

Each row represents a single task execution with the following features:
- `task_size` — size of the task workload
- `cpu_demand` — CPU resource consumption
- `memory_demand` — memory resource consumption
- `network_latency` — observed network latency
- `io_operations` — number of I/O operations
- `disk_usage` — disk resource consumption
- `num_connections` — number of active connections
- `priority_level` — scheduling priority assigned to the task
- `target` — SLA outcome (original: 1 = met, 0 = violated)
- `timestamp` — datetime of task execution

The raw dataset encodes the target as 1 = SLA met, 0 = SLA violated.
We invert this so that **1 = SLA violation** and **0 = SLA met**, which is
the standard convention for anomaly/failure detection tasks. This ensures
that all subsequent metrics (precision, recall, F1) are reported with respect
to the violation class — the outcome we care about predicting.


In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt
import shap

df = pd.read_csv('dataset.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')

# Flip target: 1 = SLA violation, 0 = SLA met
df['target'] = 1 - df['target']


## 2. Feature Engineering

Beyond the raw resource metrics, we extract 3 temporal features from the `timestamp`column:
- `hour` — hour of day (0–23), captures intra-day load patterns
- `day_of_week` — day of the week (0 = Monday, 6 = Sunday)
- `is_weekend` — binary flag for weekend vs weekday

These are included as candidate features. If the model finds no predictive signal in them, SHAP analysis will reveal this, and they can be dropped in a later iteration.

In [4]:
# Extract temporal features from timestamp
df['hour']        = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

features = [
    'task_size',
    'cpu_demand',
    'memory_demand',
    'network_latency',
    'io_operations',
    'disk_usage',
    'num_connections',
    'priority_level',
    'hour',
    'day_of_week',
    'is_weekend',
]

X = df[features]
y = df['target']


## 4. Chronolgical Train / Validation / Test Split

The dataset is split into three non-overlapping sets n strict chronological order:
- **Training set (70%)** — used to fit the model
- **Validation set (15%)** — used for threshold tuning and early stopping
- **Test set (15%)** — held out entirely; only used once for final evaluation

It is very important that we do **not** shuffle the data before splitting. Random shuffling would allow the model to train on future observations and evaluate on paste ones, producing artificially inflated metrics. Chronological splitting emulates real-world deployment, where the model must oreict future SLA violations, from patterns learnt in the past

In [5]:
train_size      = int(len(df) * 0.70)
validation_size = int(len(df) * 0.15)

X_train      = X[:train_size]
y_train      = y[:train_size]

X_validation = X[train_size : train_size + validation_size]
y_validation = y[train_size : train_size + validation_size]

X_test       = X[train_size + validation_size:]
y_test       = y[train_size + validation_size:]


## 5. Train XGBoost with Early Stopping

We use XGBoost (Extreme Gradient Boosting), a tree-based ensemble method that builds trees sequentially; each tree corrects the residual errors of the previous one. This makes it a well-suited algorithm for capturing non-linear feature interactions, that a linear model like Logistic Regression cannot represent.

Key hyperparameters:
- `max_depth=5`– limits the tree depth to prevent overfitting
- `learning_rate=0.05` – small step size; relies on more trees for accuracy
- `sumsample=0.8` – each tree is trained on 80% of rows to reduce variance
- `colsample_bytree=0.8` – each tree uses 80% of features to reduce correlation between trees
- `reg_alpha / reg_lambda` – L1 and L2 regularisation to penalise model complexity
- `early_stopping_rounds=20` – training halts if validation AUC does not improve for 20 consecutive rounds; automatically finding the optimal number of trees without manual tooning.

In [ ]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='auc',
    early_stopping_rounds=20,
    random_state=42,
    use_label_encoder=False
)

model.fit(
    X_train, y_train,
    eval_set=[(X_validation, y_validation)],
    verbose=50
)


## 6. Validation Set Evaluation

We evaluate he trained model on the valudation set using:
- **Classification report** – precision, recall and F1-score per class.
- **ROC-AUC** – measures the model's ability to rank violations above non-violations
- **Average Precision** – the area under the Precision-Recall curve, which is more informative than the ROC-AUC when the goal is to maximize recall for a specific class.

At the default threshoold of 0.5, the model achieves 94% accuracy and a ROC-AUC of 0.977%, a significant imporvement over the  Logistic Regression baseline (ROC-AUC: 0.89).

In [ ]:
val_probs = model.predict_proba(X_validation)[:, 1]
val_preds = model.predict(X_validation)

print("=== Validation Set ===")
print(classification_report(y_validation, val_preds))
print("ROC-AUC:", roc_auc_score(y_validation, val_probs))
print("Avg Precision:", average_precision_score(y_validation, val_probs))


## 7. Threshold Tuning for Maximum Recall

By default, XGBoost classifies a sample as a violation if its predicted probability
exceeds 0.5. However, because our goal is to **catch as many SLA violations as possible**,
we lower this threshold — accepting more false alarms in exchange for missing fewer
real violations.

We use the Precision-Recall curve to find the threshold that achieves approximately
98% recall on the validation set. This means the model will flag a task as a potential
violation whenever its predicted probability exceeds **0.1797**.

At this threshold:
- ~98% of actual violations are caught (only ~2% slip through)
- ~87% of alerts are genuine (1 in ~8 is a false alarm)

The appropriate threshold depends on the operational cost of each error type:
- **Missed violation (false negative)** — SLA breach goes undetected; customer impact
- **False alarm (false positive)** — unnecessary intervention; engineering cost


In [ ]:
target_recall = 0.98
precision, recall, thresholds = precision_recall_curve(y_validation, val_probs)
idx = np.argmin(np.abs(recall - target_recall))
best_threshold = thresholds[idx]
print(f"Optimal threshold for ~98% recall: {best_threshold:.4f}")
print(f"Expected Precision: {precision[idx]:.4f}")
print(f"Expected Recall:    {recall[idx]:.4f}")

val_custom = (val_probs >= best_threshold).astype(int)
print(classification_report(y_validation, val_custom))


## 8. Final Test Set Evaluation

The test set has been held out entirely until this point. It was not used during
training, validation, or threshold selection — making it a fair simulation of
how the model will perform on future, unseen data.

We apply the threshold selected in Step 7 (0.1797) to the test set predictions.
Results closely matching the validation metrics confirm that:
1. The model generalises well to unseen data
2. There is no significant overfitting to the validation set
3. The threshold selected on validation transfers reliably to new data

**Final test set performance:**
- Recall (violations): 99%
- Precision (violations): 87%
- ROC-AUC: 0.9796


In [ ]:
best_threshold = 0.1797
test_probs = model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

print("=== FINAL TEST SET ===")
print(classification_report(y_test, test_preds))
print("ROC-AUC:       ", roc_auc_score(y_test, test_probs))
print("Avg Precision: ", average_precision_score(y_test, test_probs))

## 9. SHAP Explainability

SHAP (SHapley Additive exPlanations) assigns each feature a contribution score
for every individual prediction, based on cooperative game theory. Unlike simple
feature importance metrics, SHAP values show both the **magnitude** and **direction** of each feature's influence.

The summary plot displays:
- **Y-axis** — features ranked by mean absolute SHAP value (most important at top)
- **X-axis** — SHAP value; positive = pushes toward violation, negative = pushes away
- **Colour** — feature value for that sample (red = high, blue = low)

Key findings from the SHAP analysis:
- `cpu_demand` is the dominant predictor. Counterintuitively, **low CPU demand
  pushes toward SLA violation**. This is consistent with the raw data
  (mean cpu_demand: −1.15 for violations vs +1.12 for non-violations) and
  suggests violations are driven by I/O or network-bound tasks rather than
  CPU-saturated ones — corroborated by `network_latency` and `disk_usage`
  also ranking as top features.

- `priority_level` is the second most important; low priority tasks are more
  vulnerable to SLA breaches
- `network_latency`, `disk_usage`, `task_size`, `memory_demand`, and `io_operations` each contribute moderately
- `num_connections`, `hour`, `day_of_week`, and `is_weekend` have near-zero impact,indicating no meaningful temporal pattern in SLA violations for this dataset


In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_validation)

shap.summary_plot(shap_values, X_validation, feature_names=features)


In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df['cpu_demand'], df['target'], alpha=0.1)
plt.xlabel('cpu_demand')
plt.ylabel('SLA Violation (1 = violated)')
plt.title('cpu_demand vs SLA Violation')
plt.show()


In [ ]:
df.groupby('target')['cpu_demand'].mean()


# Trade-Offs

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(recall, precision, lw=2, color='steelblue')
plt.axvline(x=0.95, color='orange', linestyle='--', label='95% recall')
plt.axvline(x=0.98, color='red',    linestyle='--', label='98% recall')
plt.xlabel('Recall (SLA Violations Caught)')
plt.ylabel('Precision (How Often Alert is Correct)')
plt.title('Precision-Recall Trade-off')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# Model Performance

## 1. Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(y_test, test_preds, display_labels=['No Violation', 'Violation'])
plt.title('Confusion Matrix (Threshold = 0.1797)')
plt.show()


## ROC Curve

In [ ]:
from sklearn.metrics import RocCurveDisplay
RocCurveDisplay.from_predictions(y_test, test_probs, name='XGBoost')
plt.title('ROC Curve')
plt.show()


## Precision-Recall Curve

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay
PrecisionRecallDisplay.from_predictions(y_test, test_probs, name='XGBoost')
plt.axvline(x=0.99, color='red', linestyle='--', label='Operating point (99% recall)')
plt.legend()
plt.show()


## SHAP Waterfall Plot – Single Prediction

In [ ]:
# Inspect the first predicted violation
idx = np.where(test_preds == 1)[0][0]
shap.plots.waterfall(shap.Explanation(
    values=shap_values[idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[idx],
    feature_names=features
))


## SHAP Dependece plot – cpu_demand

In [ ]:
shap.dependence_plot('cpu_demand', shap_values, X_validation,
                     interaction_index='priority_level')


## SHAP Force Plot – Batch View

In [ ]:
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[:200], X_validation.iloc[:200])


# Model Internals

## Feature Importance Bar Chart

In [ ]:
from xgboost import plot_importance
plot_importance(model, importance_type='gain', max_num_features=10)
plt.title('Feature Importance by Gain')
plt.show()


## Early Stopping Learning Curve

In [ ]:
results = model.evals_result()
epochs  = len(results['validation_0']['auc'])

plt.plot(results['validation_0']['auc'], label='Validation AUC')
plt.xlabel('Boosting Round')
plt.ylabel('AUC')
plt.title('XGBoost Learning Curve')
plt.axvline(x=model.best_iteration, color='red', linestyle='--', label='Best iteration')
plt.legend()
plt.show()
